# Topic Modeling
Unsupervised topic discovery on YouTube crisis comments using BERTopic.

## Methodology

### Motivation
After scoring comments by sentiment, we apply topic modeling to uncover
*what* people are talking about — blame attribution, calls for justice,
grief, misinformation, political critique — giving qualitative structure
to the quantitative sentiment scores.

### Approach
We use **BERTopic** — an embedding-based topic model that:
1. Encodes each comment into a dense vector (sentence-transformers)
2. Reduces dimensionality (UMAP) for clustering stability
3. Clusters semantically similar comments (HDBSCAN)
4. Extracts descriptive terms per cluster (c-TF-IDF)

### Pipeline

| Phase | Step | Description |
|---|---|---|
| **1. Preprocessing** | Text cleaning | Lowercase, strip punctuation/URLs/digits, remove English + Filipino stopwords |
| **2. Embedding** | Sentence vectors | `paraphrase-multilingual-MiniLM-L12-v2` (12 languages, 384-d) |
| **3. Reduction** | UMAP | 384-d → 5-d for density-based clustering |
| **4. Clustering** | HDBSCAN | Soft clustering; non-assigned → noise cluster (-1) |
| **5. Representation** | c-TF-IDF | Top unigrams/bigrams per cluster as topic descriptors |
| **6. Labeling** | Manual annotation | Human-readable labels from top terms + sample comments |

### Cross-Analysis
Topic assignments are joined with sentiment labels, shouting flags,
and video IDs to answer:
- *Which topics dominate each news source?*
- *Do certain topics carry more negative sentiment?*
- *Are shouted comments concentrated in specific topics?*

### Input / Output
- **Input:** `scored_{BATCH_NAME}.csv` from sentiment analysis
- **Output:** Topic-labeled DataFrame + per-video topic profile + topic × sentiment cross-tab

In [ ]:
import pandas as pd
import numpy as np
import ast
import os
import re
import string
import nltk

from nltk.corpus import stopwords
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

import spacy

from dotenv import load_dotenv
from google import genai
from google.genai import types

import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams["figure.dpi"] = 150

## Configuration
Batch selector. Change `BATCH_NAME` to switch between
`youtube_comments_batch_{1,2,3}.csv`.

In [ ]:
# --- CONFIG ---
BASE_DIR = "../output"
BATCH_NAME = "scored_youtube_comments_batch_1.csv"
BATCH_PATH = os.path.join(BASE_DIR, BATCH_NAME)

## Load Batch
Read the scored CSV. Each row is a cleaned comment with
`sentiment_label`, `sentiment_score`, `is_shouting`, and
`sentences`. These features feed into cross-analysis
after topic modeling.

In [ ]:
batch_df = pd.read_csv(BATCH_PATH)
print(f"Loaded {len(batch_df)} comments from {BATCH_NAME}")

# Parse the sentences column from stringified list back to Python list
batch_df['sentences'] = batch_df['sentences'].apply(ast.literal_eval)

batch_df[["text", "sentiment_label", "sentiment_score", "is_shouting"]].head()


## Stopwords

Three sources merged into one set:

| Source | Provides | Origin |
|---|---|---|
| **nltk** | ~179 English stopwords | `nltk.corpus.stopwords` |
| **calamanCy** | ~147 Tagalog stopwords | spaCy defaults from `tl_calamancy_md` |
| **Custom** | ~30 noise words | `po`, `opo`, `kasi`, `lang`, `yung`, `nung`, etc. |

Custom additions cover common comment filler that isn't in
standard stopword lists.


In [ ]:
nltk.download("stopwords", quiet=True)

en_stop = set(stopwords.words("english"))
nlp_tl = spacy.load("tl_calamancy_md")
tl_stop = nlp_tl.Defaults.stop_words

custom_stop = {
    "po", "opo", "kasi", "eh", "ah", "oh", "talaga", "naman",
    "lang", "pa", "na", "din", "rin", "raw", "daw", "yung",
    "nung", "sya", "ung", "ng", "sa", "ako", "ikaw", "tayo",
    "niya", "natin", "sila", "amin", "atin",
}

STOPWORDS = en_stop | tl_stop | custom_stop
print(f"Stopwords: {len(STOPWORDS)} total")
print(f"  English:  {len(en_stop)}")
print(f"  Tagalog:  {len(tl_stop)}")
print(f"  Custom:   {len(custom_stop)}")
print(f"Sample TL: {sorted(tl_stop)[:10]}")


## Text Cleaning

Apply a lightweight cleaning pipeline for topic modeling:

- Lowercase
- Remove URLs
- Strip punctuation
- Remove digits
- Filter stopwords + tokens shorter than 3 characters

Cleaned text is stored in a new `text_clean` column.
The original `text` column is preserved.


In [ ]:
def clean_for_topics(text: str) -> str:
    text = text.lower()
    text = re.sub(r"https?://\S+", "", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\d+", "", text)
    tokens = [w for w in text.split() if w not in STOPWORDS and len(w) > 2]
    return " ".join(tokens)

batch_df["text_clean"] = batch_df["text"].apply(clean_for_topics)
corpus = batch_df["text_clean"].tolist()

print(f"Corpus: {len(corpus)} documents")
print(f"Avg tokens/doc before: {np.mean([len(t.split()) for t in batch_df['text']]):.0f}")
print(f"Avg tokens/doc after:  {np.mean([len(t.split()) for t in corpus]):.0f}")


## Embeddings

Encode each cleaned comment into a dense vector using
`sentence-transformers` for semantic clustering.

### Model

`paraphrase-multilingual-MiniLM-L12-v2` — chosen because:
- **Multilingual**: supports English + Tagalog + code-switched text
- **384-dimensional**: compact enough for UMAP reduction
- **Sentence-level**: trained on NLI/STS pairs for semantic similarity
  (critical for BERTopic's clustering quality)


In [ ]:
EMBEDDING_MODEL = "paraphrase-multilingual-MiniLM-L12-v2"
embedding_model = SentenceTransformer(EMBEDDING_MODEL)

print(f"Encoding {len(corpus)} documents with {EMBEDDING_MODEL}...")
embeddings = embedding_model.encode(
    corpus,
    show_progress_bar=True,
    batch_size=64,
)
print(f"Embedding shape: {embeddings.shape}")


## BERTopic Model

Fit the topic model on the precomputed embeddings.

### Pipeline

| Step | Component | What it does |
|---|---|---|
| **1. Reduce** | UMAP (n_neighbors=15, n_components=5) | 384-d → 5-d for clustering stability |
| **2. Cluster** | HDBSCAN (min_cluster_size=10) | Density-based; unassigned → noise (-1) |
| **3. Represent** | c-TF-IDF | Top terms per cluster as topic descriptors |

### Why HDBSCAN over K-means?

| K-means | HDBSCAN (chosen) |
|---|---|
| Must set `k` upfront | Automatic — finds dense regions |
| Forces every point into a cluster | Noise cluster for off-topic comments |
| Spherical clusters only | Any shape (density-connected) |

For crisis comments, we don’t know the number of themes ahead of time.
HDBSCAN naturally finds them and isolates noise (spam, single-word comments).

In [ ]:
topic_model = BERTopic(
    embedding_model=embedding_model,
    min_topic_size=10,
    nr_topics="auto",
    calculate_probabilities=True,
    verbose=True,
)

topics, probs = topic_model.fit_transform(corpus, embeddings)

n_topics = len(set(topics)) - (1 if -1 in topics else 0)
n_noise = sum(1 for t in topics if t == -1)
print(f"\nTopics found: {n_topics}")
print(f"Noise cluster (-1): {n_noise} comments ({n_noise / len(topics):.1%})")


In [ ]:
topic_model.get_topic_info().head(15)


In [ ]:
topic_model.visualize_barchart(top_n_topics=10, n_words=10)


In [ ]:
topic_model.visualize_distribution(probs, min_probability=0.05)


## Topic Labeling

Each numeric topic cluster is assigned a 2-4 word human-readable label
using two independent methods, then reconciled.

### Method 1 — Manual (Ground Truth)
Top 10 c-TF-IDF terms + 5 representative comments per topic are printed.
Labels are assigned based on domain understanding of the crisis context.

### Method 2 — LLM (Gemma 4 31B via API)
A structured prompt is sent to `gemma-4-31b-it` through the Google Gemini
API. Each call returns a short label from the topic’s top terms and samples.

### Reconciliation
Where manual and LLM agree, the label is used. Where they differ,
a review determines the more accurate label.


In [ ]:
for topic_id in sorted([t for t in set(topics) if t != -1]):
    terms = topic_model.get_topic(topic_id)
    term_str = ", ".join([t[0] for t in terms[:10]])

    mask = [t == topic_id for t in topics]
    samples = batch_df.loc[mask, "text"].sample(min(5, sum(mask))).tolist()

    print(f"\n=== Topic {topic_id} ===")
    print(f"  Top terms: {term_str}")
    for s in samples:
        print(f"  · {s[:150]}")

# --- EDIT THIS DICT ---
TOPIC_LABELS = {
    -1: "Noise / Unclassified",
    # 0: "...",
}
TOPIC_LABELS


In [ ]:
load_dotenv("../../.env")
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

MODEL = "gemma-4-31b-it"

LLM_LABELS = {}
for topic_id in sorted([t for t in set(topics) if t != -1]):
    terms = topic_model.get_topic(topic_id)
    term_str = ", ".join([t[0] for t in terms[:10]])

    mask = [t == topic_id for t in topics]
    samples = batch_df.loc[mask, "text"].sample(min(5, sum(mask))).tolist()

    prompt = (
        "You are a crisis discourse analyst. Given the top terms "
        "and sample comments for a topic cluster, assign a short "
        "label (2-4 words).\n\n"
        f"Top terms: {term_str}\n"
        "Sample comments:\n"
        + "\n".join(f"- {s[:200]}" for s in samples[:5])
        + "\n\nLabel (2-4 words):"
    )

    response = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(temperature=0),
    )
    label = response.text.strip()
    LLM_LABELS[topic_id] = label
    print(f"Topic {topic_id}: {label}")


In [ ]:
comparison = pd.DataFrame({
    "topic_id": [k for k in TOPIC_LABELS if k != -1],
})
comparison["manual"] = comparison["topic_id"].map(TOPIC_LABELS)
comparison["llm"] = comparison["topic_id"].map(LLM_LABELS)
comparison["match"] = comparison["manual"] == comparison["llm"]

print(f"Agreement: {comparison['match'].mean():.0%}")
print("\nMismatches:")
comparison[~comparison["match"]]


In [ ]:
batch_df["topic_id"] = topics
batch_df["topic_label"] = batch_df["topic_id"].map(TOPIC_LABELS)
batch_df[["text_clean", "topic_id", "topic_label"]].head(10)


## Cross-Analysis

Topic labels are joined with sentiment, shouting, and video metadata
to answer the research questions from the methodology:

- *Which topics dominate each news source (video_id)?*
  → Per-video topic distribution (crosstab + stacked bar)

- *Do certain topics carry more negative sentiment?*
  → Topic × sentiment_label heatmap

- *Are shouted comments concentrated in specific topics?*
  → Shouting ratio per topic


In [ ]:
topic_x_video = pd.crosstab(
    batch_df["topic_label"],
    batch_df["video_id"],
    margins=True,
    margins_name="Total",
)
topic_x_video


In [ ]:
topic_x_sentiment = pd.crosstab(
    batch_df["topic_label"],
    batch_df["sentiment_label"],
    margins=True,
    margins_name="Total",
)
topic_x_sentiment


In [ ]:
shouting_by_topic = batch_df.groupby("topic_label")["is_shouting"].agg(
    total="count",
    shouted="sum",
)
shouting_by_topic["shouting_ratio"] = shouting_by_topic["shouted"] / shouting_by_topic["total"]
shouting_by_topic.sort_values("shouting_ratio", ascending=False)


In [ ]:
topic_video_pct = (
    batch_df.groupby(["video_id", "topic_label"])
    .size()
    .unstack(fill_value=0)
)
topic_video_pct = topic_video_pct.div(topic_video_pct.sum(axis=1), axis=0)

ax = topic_video_pct.plot(
    kind="bar",
    stacked=True,
    figsize=(10, 5),
    colormap="tab10",
)
plt.ylabel("Proportion")
plt.title("Topic Composition per Video")
plt.legend(title="Topic", bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.show()


## Export

Save the topic-labeled comments for downstream use or
comparison across batches.

Columns preserved: text, video_id, sentiment_label, sentiment_score,
is_shouting, topic_id, topic_label, sentences.


In [ ]:
output_dir = "../output"
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, f"topics_{BATCH_NAME.replace('scored_', '')}")
batch_df.to_csv(output_path, index=False)
print(f"Exported {len(batch_df)} topic-labeled comments to {output_path}")
